# BEATs 四数据集 M-Unified（seed 42）

本Notebook复用已完成的四数据集ground-truth ledger、ontology、eligibility、split、层级head/loss与metrics。唯一package变化是frozen encoder/frontend从AST切换为BEATs；四个native prediction units不统一。

## BEATs package定义

每个16 kHz、2秒source window进入本地AudioSet pretrained BEATs。实现使用真实valid samples推导exact flattened patch mask，在Transformer后恢复time-major/frequency-minor patch grid，先对frequency patches取mean，再只对有效temporal tokens取mean，得到每窗口768维embedding。encoder冻结；cache不复用AST embeddings。

结果只能解释为 `BEATs_exact_valid_patch_pool_v0` 的encoder+frontend+masking+pooling package，不称纯backbone gain。

## 冻结训练与评估合同

seed=42；2秒/1秒；homogeneous native batch=8；source-proportional；50 epochs；Adam lr=5e-5、weight decay=1e-6；cosine per update、no warmup；FP32、无augmentation。Level1 CE；Crackle/Wheeze/Other在eligible rows上BCE，各node等权。每epoch validation，best/last checkpoint；validation loss选模、validation max-F1定threshold后，test只运行一次。

In [ ]:
from pathlib import Path
import json
from baseline.multidataset_pipeline.m_unified import run_beats_m_unified

ROOT = Path.cwd()
RESULT = ROOT / 'result/reproduce/unified/BEATs_M_Unified/seed_42'
summary_path = RESULT / 'run_summary.json'
summary = json.loads(summary_path.read_text()) if summary_path.is_file() else run_beats_m_unified(
    ROOT, RESULT, device_name='cpu', encoder_window_batch_size=8
)
summary

主结果按dataset/node分别报告metrics、confusion与support；dataset-macro与worst-dataset保留support caveat。HF仍是positive-only；KAUH unresolved仍mask；combined pooled matrix仅辅助。